# CrisisMap ML Fine-Tuning — Stage B: Beirut Domain Calibration

**Objective:** Calibrate the Stage A fine-tuned SiamUnet to Beirut Maxar GeoTIFF domain.

**Prerequisites:**
- Stage A checkpoint (`model_best.pth.tar` from `01_finetune_xbd.ipynb`)
- Beirut Maxar pre/post GeoTIFF imagery
- Building footprints shapefile
- Copernicus EMSR452 ground truth labels

**POIN 6 — Anti-Leakage:**
- Held-out test set NEVER used for calibration
- No overlap with 60 buildings from ASM-D06926 validation set
- 'Possibly Damaged' buildings EXCLUDED from training (marked uncertain)

**Strategy:**
- Load best Stage A checkpoint
- Freeze everything except damage classifier
- Fine-tune on Beirut chip pairs with micro learning rate (1e-5)
- Report F1 ONLY on held-out test set

In [ ]:
# ============================================================================
# CELL 0: Environment Setup
# ============================================================================
import os
import sys
import json
import time
import random
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')

In [ ]:
# ============================================================================
# CELL 1: Paths & Configuration
# ============================================================================

IS_KAGGLE = os.path.exists('/kaggle/input')

if IS_KAGGLE:
    STAGE_A_CHECKPOINT = Path('/kaggle/input/crisismap-stage-a/model_best.pth.tar')
    BEIRUT_PRE = Path('/kaggle/input/crisismap-beirut/pre_event.tif')
    BEIRUT_POST = Path('/kaggle/input/crisismap-beirut/post_event.tif')
    FOOTPRINTS_PATH = Path('/kaggle/input/crisismap-beirut/footprints.shp')
    EMSR452_PATH = Path('/kaggle/input/crisismap-beirut/EMSR452_ground_truth.json')
    OUTPUT_DIR = Path('/kaggle/working/output_stage_b')
else:
    STAGE_A_CHECKPOINT = Path(r'c:/Users/HP/Laravel/CrisisMap/ml-training/notebooks/output/model_best.pth.tar')
    BEIRUT_PRE = Path(r'c:/Users/HP/Laravel/CrisisMap/data/raw/imagery/maxar/beirut_explosion/pre_event_2020-07-31_10300500A5F95600.tif')
    BEIRUT_POST = Path(r'c:/Users/HP/Laravel/CrisisMap/data/raw/imagery/maxar/beirut_explosion/post_event_2020-08-05_104001005EBCEB00.tif')
    FOOTPRINTS_PATH = Path(r'c:/Users/HP/Laravel/CrisisMap/data/raw/vector/maxar/beirut_explosion/extracted/Beirut-Explosion-2D-building-32636/Beirut-Explosion-2D-building-32636.shp')
    EMSR452_PATH = Path(r'c:/Users/HP/Laravel/CrisisMap/data/raw/ground_truth/copernicus_ems/EMSR452/extracted/EMSR452_AOI01_GRA_PRODUCT_builtUpP_r1_v2.json')
    OUTPUT_DIR = Path('./output_stage_b')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# EMSR452 damage grade -> xBD class mapping (POIN 6)
EMSR452_MAPPING = {
    'Destroyed': 4,          # -> destroyed
    'Damaged': 3,            # -> major-damage
    # 'Possibly Damaged' is EXCLUDED — uncertain, do NOT force to minor-damage
}
DAMAGE_CLASSES = ['background', 'no-damage', 'minor-damage', 'major-damage', 'destroyed']

# Stage B hyperparameters
STAGE_B_CONFIG = {
    'lr': 1e-5,
    'weight_decay': 1e-5,
    'epochs': 10,
    'batch_size': 8,
    'num_workers': 2,
    'class_weights_dmg': [0.05, 0.20, 0.60, 0.80, 1.00],
    'test_split_ratio': 0.25,  # 25% held-out for unbiased evaluation
}

print(f'Stage A checkpoint: {STAGE_A_CHECKPOINT} (exists: {STAGE_A_CHECKPOINT.exists()})')
print(f'Beirut pre:  {BEIRUT_PRE} (exists: {BEIRUT_PRE.exists()})')
print(f'Beirut post: {BEIRUT_POST} (exists: {BEIRUT_POST.exists()})')
print(f'Footprints:  {FOOTPRINTS_PATH} (exists: {FOOTPRINTS_PATH.exists()})')
print(f'EMSR452:     {EMSR452_PATH} (exists: {EMSR452_PATH.exists()})')

In [ ]:
# ============================================================================
# CELL 2: SiamUnet Architecture (same as Stage A — DO NOT MODIFY)
# ============================================================================
from collections import OrderedDict

class SiamUnet(nn.Module):
    def __init__(self, in_channels=3, out_channels_s=2, out_channels_c=5, init_features=16):
        super(SiamUnet, self).__init__()
        features = init_features
        self.encoder1 = SiamUnet._block(in_channels, features, name='enc1')
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.encoder2 = SiamUnet._block(features, features * 2, name='enc2')
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.encoder3 = SiamUnet._block(features * 2, features * 4, name='enc3')
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.encoder4 = SiamUnet._block(features * 4, features * 8, name='enc4')
        self.pool4 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.bottleneck = SiamUnet._block(features * 8, features * 16, name='bottleneck')
        self.upconv4 = nn.ConvTranspose2d(features * 16, features * 8, kernel_size=2, stride=2)
        self.decoder4 = SiamUnet._block((features * 8) * 2, features * 8, name='dec4')
        self.upconv3 = nn.ConvTranspose2d(features * 8, features * 4, kernel_size=2, stride=2)
        self.decoder3 = SiamUnet._block((features * 4) * 2, features * 4, name='dec3')
        self.upconv2 = nn.ConvTranspose2d(features * 4, features * 2, kernel_size=2, stride=2)
        self.decoder2 = SiamUnet._block((features * 2) * 2, features * 2, name='dec2')
        self.upconv1 = nn.ConvTranspose2d(features * 2, features, kernel_size=2, stride=2)
        self.decoder1 = SiamUnet._block(features * 2, features, name='dec1')
        self.conv_s = nn.Conv2d(in_channels=features, out_channels=out_channels_s, kernel_size=1)
        self.upconv4_c = nn.ConvTranspose2d(features * 16, features * 8, kernel_size=2, stride=2)
        self.conv4_c = SiamUnet._block(features * 16, features * 16, name='conv4')
        self.upconv3_c = nn.ConvTranspose2d(features * 16, features * 4, kernel_size=2, stride=2)
        self.conv3_c = SiamUnet._block(features * 8, features * 8, name='conv3')
        self.upconv2_c = nn.ConvTranspose2d(features * 8, features * 2, kernel_size=2, stride=2)
        self.conv2_c = SiamUnet._block(features * 4, features * 4, name='conv2')
        self.upconv1_c = nn.ConvTranspose2d(features * 4, features, kernel_size=2, stride=2)
        self.conv1_c = SiamUnet._block(features * 2, features * 2, name='conv1')
        self.conv_c = nn.Conv2d(in_channels=features * 2, out_channels=out_channels_c, kernel_size=1)

    def forward(self, x1, x2):
        enc1_1 = self.encoder1(x1)
        enc2_1 = self.encoder2(self.pool1(enc1_1))
        enc3_1 = self.encoder3(self.pool2(enc2_1))
        enc4_1 = self.encoder4(self.pool3(enc3_1))
        bottleneck_1 = self.bottleneck(self.pool4(enc4_1))
        dec4_1 = self.decoder4(torch.cat((self.upconv4(bottleneck_1), enc4_1), dim=1))
        dec3_1 = self.decoder3(torch.cat((self.upconv3(dec4_1), enc3_1), dim=1))
        dec2_1 = self.decoder2(torch.cat((self.upconv2(dec3_1), enc2_1), dim=1))
        dec1_1 = self.decoder1(torch.cat((self.upconv1(dec2_1), enc1_1), dim=1))
        enc1_2 = self.encoder1(x2)
        enc2_2 = self.encoder2(self.pool1(enc1_2))
        enc3_2 = self.encoder3(self.pool2(enc2_2))
        enc4_2 = self.encoder4(self.pool3(enc3_2))
        bottleneck_2 = self.bottleneck(self.pool4(enc4_2))
        dec4_2 = self.decoder4(torch.cat((self.upconv4(bottleneck_2), enc4_2), dim=1))
        dec3_2 = self.decoder3(torch.cat((self.upconv3(dec4_2), enc3_2), dim=1))
        dec2_2 = self.decoder2(torch.cat((self.upconv2(dec3_2), enc2_2), dim=1))
        dec1_2 = self.decoder1(torch.cat((self.upconv1(dec2_2), enc1_2), dim=1))
        dec1_c = bottleneck_2 - bottleneck_1
        dec2_c = self.conv4_c(torch.cat((enc4_2 - enc4_1, self.upconv4_c(dec1_c)), dim=1))
        dec3_c = self.conv3_c(torch.cat((enc3_2 - enc3_1, self.upconv3_c(dec2_c)), dim=1))
        dec4_c = self.conv2_c(torch.cat((enc2_2 - enc2_1, self.upconv2_c(dec3_c)), dim=1))
        dec5_c = self.conv1_c(torch.cat((enc1_2 - enc1_1, self.upconv1_c(dec4_c)), dim=1))
        return self.conv_s(dec1_1), self.conv_s(dec1_2), self.conv_c(dec5_c)

    @staticmethod
    def _block(in_channels, features, name):
        return nn.Sequential(OrderedDict([
            (name + 'conv1', nn.Conv2d(in_channels, features, kernel_size=3, padding=1, bias=False)),
            (name + 'norm1', nn.BatchNorm2d(features)),
            (name + 'relu1', nn.ReLU(inplace=True)),
            (name + 'conv2', nn.Conv2d(features, features, kernel_size=3, padding=1, bias=False)),
            (name + 'norm2', nn.BatchNorm2d(features)),
            (name + 'relu2', nn.ReLU(inplace=True)),
        ]))

print('SiamUnet architecture loaded.')

In [ ]:
# ============================================================================
# CELL 3: Load EMSR452 Ground Truth & Generate Beirut Chip Pairs
# ============================================================================

def load_emsr452_buildings(emsr_path, mapping):
    """Load EMSR452 ground truth and filter to usable buildings.
    
    POIN 6: 'Possibly Damaged' buildings are EXCLUDED (marked uncertain).
    Only 'Destroyed' and 'Damaged' are used.
    """
    with open(emsr_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    features = data.get('features', [])
    buildings = []
    excluded = 0
    
    for feat in features:
        props = feat.get('properties', {})
        geom = feat.get('geometry', {})
        damage_grade = str(props.get('damage_gra', '')).strip()
        
        if damage_grade not in mapping:
            excluded += 1
            continue
        
        if geom.get('type') != 'Point':
            continue
        
        coords = geom.get('coordinates', [None, None])
        if coords[0] is None:
            continue
        
        buildings.append({
            'lon': coords[0],
            'lat': coords[1],
            'damage_grade': damage_grade,
            'damage_class': mapping[damage_grade],
            'properties': props,
        })
    
    print(f'EMSR452 buildings loaded: {len(buildings)} usable, {excluded} excluded (Possibly Damaged)')
    for grade, cls_idx in mapping.items():
        n = sum(1 for b in buildings if b['damage_grade'] == grade)
        print(f'  {grade}: {n} buildings -> class {cls_idx} ({DAMAGE_CLASSES[cls_idx]})')
    
    return buildings


def generate_beirut_chips(buildings, pre_path, post_path, chip_size=256, buffer_deg=0.0015):
    """Generate chip pairs from Beirut GeoTIFF for each building.
    
    Preprocessing matches backend ml_inference.py:
      - src.read(indexes=[1,2,3]) -> RGB
      - Resize to (3, 256, 256)
      - scale = max(p99, 255.0)
      - data = clip(data / scale, 0, 1)
    """
    import rasterio
    from rasterio.windows import from_bounds
    
    chips = []
    
    with rasterio.open(pre_path) as pre_src, rasterio.open(post_path) as post_src:
        for bld in tqdm(buildings, desc='Generating chips'):
            lon, lat = bld['lon'], bld['lat']
            bbox = (lon - buffer_deg, lat - buffer_deg, lon + buffer_deg, lat + buffer_deg)
            
            try:
                # Read pre chip
                pre_window = from_bounds(*bbox, transform=pre_src.transform)
                pre_data = pre_src.read(
                    indexes=[1, 2, 3],
                    window=pre_window,
                    out_shape=(3, chip_size, chip_size),
                    boundless=True, fill_value=0
                ).astype(np.float32)
                
                # Read post chip
                post_window = from_bounds(*bbox, transform=post_src.transform)
                post_data = post_src.read(
                    indexes=[1, 2, 3],
                    window=post_window,
                    out_shape=(3, chip_size, chip_size),
                    boundless=True, fill_value=0
                ).astype(np.float32)
                
                # Normalize (match backend exactly)
                pre_scale = max(float(np.percentile(pre_data, 99)), 255.0)
                pre_data = np.clip(pre_data / pre_scale, 0.0, 1.0)
                
                post_scale = max(float(np.percentile(post_data, 99)), 255.0)
                post_data = np.clip(post_data / post_scale, 0.0, 1.0)
                
                # Skip blank chips
                if pre_data.max() < 0.01 or post_data.max() < 0.01:
                    continue
                
                chips.append({
                    'pre': pre_data,
                    'post': post_data,
                    'damage_class': bld['damage_class'],
                    'damage_grade': bld['damage_grade'],
                    'lon': lon,
                    'lat': lat,
                })
            except Exception as e:
                continue
    
    print(f'Generated {len(chips)} valid chip pairs from {len(buildings)} buildings')
    return chips


# Load ground truth
if EMSR452_PATH.exists():
    gt_buildings = load_emsr452_buildings(EMSR452_PATH, EMSR452_MAPPING)
else:
    print(f'WARNING: EMSR452 file not found at {EMSR452_PATH}')
    print('Stage B requires ground truth. Please provide the file.')
    gt_buildings = []

In [ ]:
# ============================================================================
# CELL 4: Generate Chips & Split (Anti-Leakage)
# ============================================================================

if gt_buildings and BEIRUT_PRE.exists() and BEIRUT_POST.exists():
    chips = generate_beirut_chips(gt_buildings, BEIRUT_PRE, BEIRUT_POST)
else:
    print('Skipping chip generation (missing data files).')
    print('To run Stage B, ensure Beirut imagery and EMSR452 ground truth are available.')
    chips = []

if chips:
    # POIN 6: Anti-leakage split
    # Held-out test set NEVER used for calibration
    rng = np.random.RandomState(SEED)
    indices = np.arange(len(chips))
    rng.shuffle(indices)
    
    n_test = max(5, int(len(chips) * STAGE_B_CONFIG['test_split_ratio']))
    n_cal = len(chips) - n_test
    
    cal_indices = indices[:n_cal]
    test_indices = indices[n_cal:]
    
    cal_chips = [chips[i] for i in cal_indices]
    test_chips = [chips[i] for i in test_indices]
    
    print(f'\nAnti-leakage split:')
    print(f'  Calibration: {len(cal_chips)} chips (for fine-tuning)')
    print(f'  Test:        {len(test_chips)} chips (held-out, NEVER used for training)')
    
    # Distribution
    for name, subset in [('Calibration', cal_chips), ('Test', test_chips)]:
        dist = defaultdict(int)
        for c in subset:
            dist[c['damage_grade']] += 1
        print(f'  {name} distribution: {dict(dist)}')

In [ ]:
# ============================================================================
# CELL 5: Beirut Chip Dataset
# ============================================================================

class BeirutChipDataset(Dataset):
    """Dataset for Beirut calibration chips.
    
    Building-level labels: each chip gets a single damage class.
    The entire chip is filled with that class for pixel-level loss.
    """
    
    def __init__(self, chips, chip_size=256, augment=False):
        self.chips = chips
        self.chip_size = chip_size
        self.augment = augment
    
    def __len__(self):
        return len(self.chips)
    
    def __getitem__(self, idx):
        chip = self.chips[idx]
        pre = chip['pre'].copy()   # (3, 256, 256)
        post = chip['post'].copy()
        damage_class = chip['damage_class']
        
        # Create uniform damage mask (building-level label)
        damage_mask = np.full((self.chip_size, self.chip_size), damage_class, dtype=np.int64)
        seg_mask = np.ones((self.chip_size, self.chip_size), dtype=np.int64)  # all building
        
        if self.augment:
            if random.random() > 0.5:
                pre = np.flip(pre, axis=2).copy()
                post = np.flip(post, axis=2).copy()
            if random.random() > 0.5:
                pre = np.flip(pre, axis=1).copy()
                post = np.flip(post, axis=1).copy()
        
        return {
            'pre': torch.from_numpy(pre).float(),
            'post': torch.from_numpy(post).float(),
            'seg_pre': torch.from_numpy(seg_mask).long(),
            'seg_post': torch.from_numpy(seg_mask).long(),
            'damage': torch.from_numpy(damage_mask).long(),
        }

if chips:
    cal_dataset = BeirutChipDataset(cal_chips, augment=True)
    test_dataset_b = BeirutChipDataset(test_chips, augment=False)
    
    cal_loader = DataLoader(cal_dataset, batch_size=STAGE_B_CONFIG['batch_size'], shuffle=True, num_workers=0)
    test_loader_b = DataLoader(test_dataset_b, batch_size=STAGE_B_CONFIG['batch_size'], shuffle=False, num_workers=0)
    
    print(f'Calibration loader: {len(cal_loader)} batches')
    print(f'Test loader: {len(test_loader_b)} batches')

In [ ]:
# ============================================================================
# CELL 6: Load Stage A Checkpoint & Freeze Layers
# ============================================================================

model = SiamUnet()

if STAGE_A_CHECKPOINT.exists():
    ckpt = torch.load(STAGE_A_CHECKPOINT, map_location='cpu', weights_only=False)
    state_dict = ckpt.get('state_dict', ckpt)
    # Strip module./model. prefixes
    cleaned = {}
    for k, v in state_dict.items():
        k_clean = k
        for prefix in ('module.', 'model.'):
            if k_clean.startswith(prefix):
                k_clean = k_clean[len(prefix):]
        cleaned[k_clean] = v
    result = model.load_state_dict(cleaned, strict=True)
    print(f'Stage A checkpoint loaded: epoch={ckpt.get("epoch", "?")}, best_f1={ckpt.get("best_f1", "?")}')
    print(f'  Missing keys: {result.missing_keys}')
else:
    print(f'WARNING: Stage A checkpoint not found at {STAGE_A_CHECKPOINT}')
    print('Using random weights. Run Stage A first!')

model = model.to(DEVICE)

# Freeze everything EXCEPT damage classifier (same as Stage A Phase 2)
SEG_LAYER_PREFIXES = [
    'encoder1', 'encoder2', 'encoder3', 'encoder4',
    'bottleneck', 'pool',
    'upconv4.', 'decoder4', 'upconv3.', 'decoder3',
    'upconv2.', 'decoder2', 'upconv1.', 'decoder1',
    'conv_s',
]

frozen = 0
for name, param in model.named_parameters():
    if any(name.startswith(p) for p in SEG_LAYER_PREFIXES):
        param.requires_grad = False
        frozen += 1
    else:
        param.requires_grad = True

trainable = sum(1 for p in model.parameters() if p.requires_grad)
print(f'Frozen: {frozen} params, Trainable: {trainable} params (damage classifier only)')

In [ ]:
# ============================================================================
# CELL 7: Stage B Training Loop
# ============================================================================

if chips:
    dmg_weights = torch.tensor(STAGE_B_CONFIG['class_weights_dmg'], dtype=torch.float32).to(DEVICE)
    dmg_criterion = nn.CrossEntropyLoss(weight=dmg_weights)
    
    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=STAGE_B_CONFIG['lr'],
        weight_decay=STAGE_B_CONFIG['weight_decay'],
    )
    
    print('='*60)
    print('STAGE B: Beirut Domain Calibration')
    print(f'  LR: {STAGE_B_CONFIG["lr"]}, Epochs: {STAGE_B_CONFIG["epochs"]}')
    print(f'  Calibration chips: {len(cal_chips)}')
    print(f'  Test chips (held-out): {len(test_chips)}')
    print('='*60)
    
    history_b = {'train_loss': [], 'test_f1': []}
    best_test_f1 = 0.0
    
    for epoch in range(1, STAGE_B_CONFIG['epochs'] + 1):
        # Train
        model.train()
        running_loss = 0.0
        n_batches = 0
        
        for batch in cal_loader:
            pre = batch['pre'].to(DEVICE)
            post = batch['post'].to(DEVICE)
            damage_gt = batch['damage'].to(DEVICE)
            
            optimizer.zero_grad()
            _, _, damage_pred = model(pre, post)
            loss = dmg_criterion(damage_pred, damage_gt)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            n_batches += 1
        
        avg_loss = running_loss / max(n_batches, 1)
        
        # Evaluate on HELD-OUT test set (POIN 6)
        model.eval()
        all_pred = []
        all_true = []
        
        with torch.no_grad():
            for batch in test_loader_b:
                pre = batch['pre'].to(DEVICE)
                post = batch['post'].to(DEVICE)
                damage_gt = batch['damage']
                
                _, _, damage_pred = model(pre, post)
                # Building-level: average softmax over entire chip (matches backend)
                probs = torch.softmax(damage_pred, dim=1).mean(dim=(2, 3))  # (B, 5)
                pred_cls = probs.argmax(dim=1).cpu().numpy()
                true_cls = damage_gt[:, 0, 0].numpy()  # uniform label
                
                all_pred.extend(pred_cls.tolist())
                all_true.extend(true_cls.tolist())
        
        # Building-level F1 (POIN 5)
        all_pred = np.array(all_pred)
        all_true = np.array(all_true)
        
        f1_vals = []
        for cls in [3, 4]:  # major-damage, destroyed
            tp = ((all_pred == cls) & (all_true == cls)).sum()
            fp = ((all_pred == cls) & (all_true != cls)).sum()
            fn = ((all_pred != cls) & (all_true == cls)).sum()
            p = tp / (tp + fp) if (tp + fp) > 0 else 0
            r = tp / (tp + fn) if (tp + fn) > 0 else 0
            f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0
            f1_vals.append(f1)
        
        macro_f1 = float(np.mean(f1_vals)) if f1_vals else 0.0
        accuracy = float((all_pred == all_true).sum() / len(all_true)) if len(all_true) > 0 else 0
        
        history_b['train_loss'].append(avg_loss)
        history_b['test_f1'].append(macro_f1)
        
        is_best = macro_f1 > best_test_f1
        if is_best:
            best_test_f1 = macro_f1
            # Save best checkpoint
            torch.save({
                'epoch': epoch,
                'state_dict': model.state_dict(),
                'optimizer': optimizer.state_dict(),
                'val_f1_avg': macro_f1,
                'best_f1': best_test_f1,
            }, OUTPUT_DIR / 'model_best_beirut.pth.tar')
        
        print(f'Epoch {epoch:2d}/{STAGE_B_CONFIG["epochs"]} | '
              f'Loss: {avg_loss:.4f} | '
              f'Test F1: {macro_f1:.4f} | '
              f'Test Acc: {accuracy:.4f}'
              f'{" *BEST*" if is_best else ""}')
    
    print(f'\nStage B complete. Best test F1: {best_test_f1:.4f}')
else:
    print('Stage B skipped (no chip data available).')

In [ ]:
# ============================================================================
# CELL 8: Final Checkpoint Compatibility Test
# ============================================================================

best_path = OUTPUT_DIR / 'model_best_beirut.pth.tar'

if best_path.exists():
    ckpt = torch.load(best_path, map_location='cpu', weights_only=False)
    test_model = SiamUnet()
    result = test_model.load_state_dict(ckpt['state_dict'], strict=True)
    assert len(result.missing_keys) == 0
    
    test_model.eval()
    with torch.no_grad():
        s1, s2, d = test_model(torch.randn(1, 3, 256, 256), torch.randn(1, 3, 256, 256))
    assert s1.shape == (1, 2, 256, 256)
    assert d.shape == (1, 5, 256, 256)
    print('Checkpoint compatibility: ALL TESTS PASSED')
    print(f'Best F1: {ckpt.get("best_f1", "?")}')
    print(f'\nDeploy: copy model_best_beirut.pth.tar to backend models directory')
else:
    print('No Stage B checkpoint to verify.')